In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
expnr   = 19
out_dir = f'/Users/yunpeichu/LS2D-ICON/results/mpcseed/run_site_based_0{expnr:02d}'

prof_path = f'{out_dir}/prof.inp.{expnr:03d}'
icon_path = f'{out_dir}/CLOUDLAB_MIP_input_130_icon_les.nc'

In [ ]:
# Load Datasets

# DALES initial profile (prof.inp) — plain text, 2 header lines
# Line 1: docstring
# Line 2: column names
# Lines 3+: whitespace-separated data
try:
    with open(prof_path) as f:
        col_names = f.readline()          # skip docstring
        col_names = f.readline().split()  # parse column names
    df_prof = pd.read_csv(
        prof_path, skiprows=2, sep=r'\s+', names=col_names)
    print('DALES prof.inp loaded.')
    print('Columns:', list(df_prof.columns))
except FileNotFoundError:
    print(f'DALES prof.inp not found: {prof_path}')

# ICON LES state profiles (NetCDF)
try:
    ds_icon = xr.open_dataset(icon_path)
    print('\nICON dataset loaded.')
    print('ICON Variables:', list(ds_icon.data_vars))
except FileNotFoundError:
    print(f'ICON file not found: {icon_path}')

In [ ]:
# Quick comparison: DALES prof.inp vs ICON initial profiles
t = 0  # first time step

fig, axes = plt.subplots(1, 4, figsize=(14, 7), sharey=True)

pairs = [
    ('thl (K)',      'thl', 'thl (K)'),
    ('qt (kg kg-1)', 'qt',  'qt (kg kg-1)'),
    ('u (m s-1)',    'u',   'u (m s-1)'),
    ('v (m s-1)',    'v',   'v (m s-1)'),
]

for ax, (prof_col, icon_var, label) in zip(axes, pairs):
    ax.plot(df_prof[prof_col],  df_prof['z (m)'], label='DALES prof.inp', lw=1.5)
    if icon_var in ds_icon:
        ax.plot(ds_icon[icon_var].isel(time=t), ds_icon['z'], '--', label='ICON', lw=1.5)
    ax.set_xlabel(label)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('z (m)')
fig.suptitle(f'DALES vs ICON initial profiles  |  exp {expnr:03d}')
plt.tight_layout()
plt.show()